# RetailMind AI — Day 8: Segmentation & End-to-End CV Explainability
**Objective**: Remove noisy backgrounds for clean catalog consistency and use Grad-CAM to visualize what pixels drive the CNN's decisions.

In [ ]:
import cv2
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from tensorflow.keras.datasets import fashion_mnist

# Load a sample shoe for demonstration
(_, _), (test_images, test_labels) = fashion_mnist.load_data()
shoe_img = test_images[9] # Ankle boot
shoe_tensor = (shoe_img / 255.0).reshape(1, 28, 28, 1)
class_names = ['T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat', 'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']

In [ ]:
# 1. OpenCV Background Segmentation
_, mask = cv2.threshold(shoe_img, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
clean_bg = np.ones_like(shoe_img) * 255
product_cutout = cv2.bitwise_and(shoe_img, shoe_img, mask=mask)
clean_bg[mask > 0] = product_cutout[mask > 0]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(8, 4))
ax1.imshow(shoe_img, cmap='gray'); ax1.set_title('Original')
ax2.imshow(clean_bg, cmap='gray'); ax2.set_title('Segmented (Clean BG)')
plt.show()

In [ ]:
# 2. Grad-CAM Explainability
model = tf.keras.models.load_model('../models/catalog_cnn_v6.keras')
last_conv = [l.name for l in model.layers if isinstance(l, tf.keras.layers.Conv2D)][-1]

grad_model = tf.keras.models.Model([model.inputs], [model.get_layer(last_conv).output, model.output])
with tf.GradientTape() as tape:
    conv_outputs, preds = grad_model(shoe_tensor)
    class_channel = preds[:, tf.argmax(preds[0])]

grads = tape.gradient(class_channel, conv_outputs)
pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
heatmap = conv_outputs[0] @ pooled_grads[..., tf.newaxis]
heatmap = tf.maximum(tf.squeeze(heatmap), 0) / tf.math.reduce_max(heatmap)

heatmap_resized = cv2.resize(heatmap.numpy(), (28, 28))
plt.imshow(clean_bg, cmap='gray')
plt.imshow(heatmap_resized, cmap='jet', alpha=0.5)
plt.title(f'Grad-CAM: {class_names[tf.argmax(preds[0])]}')
plt.show()